# 4.4 Handling Missing Values

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.04_handling-missing-values.ipynb)

Companion code for Section 4.4 of the book. Run the cells in order from top to bottom.

**Example 4.4.1**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create a dataset with various missing patterns
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age':       np.where(np.random.rand(n)<0.05,  np.nan, np.random.normal(40,12,n).clip(18,80)),
    'income':    np.where(np.random.rand(n)<0.12,  np.nan, np.random.exponential(60000,n)),
    'bmi':       np.where(np.random.rand(n)<0.08,  np.nan, np.random.normal(25,5,n).clip(15,50)),
    'education': np.where(np.random.rand(n)<0.03,  np.nan, np.random.choice(['HS','BS','MS','PhD'],n)),
    'credit_score':np.where(np.random.rand(n)<0.20,np.nan, np.random.normal(680,80,n).clip(300,850)),
    'target':    np.random.choice([0,1],n,p=[0.7,0.3])
})

# ── Comprehensive missing value profile ──────────────────────────
def missing_profile(df):
    total = len(df)
    miss  = df.isnull().sum()
    miss_pct = miss / total * 100

    profile = pd.DataFrame({
        'missing_count': miss,
        'missing_%':     miss_pct.round(2),
        'dtype':         df.dtypes,
        'n_unique':      df.nunique(),
        'recommendation': miss_pct.apply(lambda p:
            'Drop column'   if p > 60 else
            'Impute + flag' if p > 20 else
            'Impute'        if p > 0  else
            'Complete')
    })
    return profile[miss > 0].sort_values('missing_%', ascending=False)

print('Missing Value Profile:')
print(missing_profile(df))

# ── Missingness pattern analysis ──────────────────────────────────
# Are missing values correlated across columns?
miss_matrix = df.isnull().astype(int)
miss_corr   = miss_matrix.corr()
print('\nMissingness correlation (do cols go missing together?):')
print(miss_corr.round(3))

# ── Visualise missing pattern (missingno-style) ───────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of missingness
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', ax=ax1, yticklabels=False)
ax1.set_title('Missing Value Pattern\n(yellow = missing)'); ax1.set_xlabel('Features')

# Bar chart of missing percentages
miss_pct = df.isnull().mean().mul(100).sort_values(ascending=True)
miss_pct[miss_pct > 0].plot(kind='barh', ax=ax2, color='#E53935', alpha=0.8)
ax2.axvline(x=5,  color='orange', linestyle='--', label='5% threshold')
ax2.axvline(x=20, color='red',    linestyle='--', label='20% threshold')
ax2.set_xlabel('Missing %'); ax2.set_title('Missing Value Percentages')
ax2.legend()

plt.tight_layout()
plt.savefig('missing_values.png', dpi=150, bbox_inches='tight')
print('Missing value chart saved.')

**Example 4.4.2**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.experimental import enable_iterative_imputer

# ── Simple Imputation ─────────────────────────────────────────────
# Mean imputation for numeric
mean_imputer   = SimpleImputer(strategy='mean')
median_imputer = SimpleImputer(strategy='median')
mode_imputer   = SimpleImputer(strategy='most_frequent')

# ── KNN Imputation - uses k nearest neighbours ────────────────────
knn_imputer = KNNImputer(n_neighbors=5, weights='distance')

# ── Iterative Imputation (MICE) ───────────────────────────────────
iter_imputer = IterativeImputer(
    max_iter=10,
    random_state=42,
    estimator=None   # Default: BayesianRidge; can use RandomForest
)

# Compare imputation methods on numeric data
np.random.seed(42)
X_complete = np.random.normal([0, 2, -1], [1, 1.5, 0.8], size=(200, 3))
X_complete[:, 1] += 0.5 * X_complete[:, 0]  # correlation between features

# Introduce 15% missing at random
X_missing = X_complete.copy()
mask = np.random.rand(200, 3) < 0.15
X_missing[mask] = np.nan
print(f'Missing values: {mask.sum()} ({mask.mean():.1%})')

# Apply each imputer and measure error
methods = {
    'Mean':      SimpleImputer(strategy='mean'),
    'Median':    SimpleImputer(strategy='median'),
    'KNN(k=5)':  KNNImputer(n_neighbors=5),
    'Iterative': IterativeImputer(max_iter=10, random_state=42),
}

print(f'\n{"Method":<14} {"RMSE on missing values"}')
print('-' * 40)
for name, imputer in methods.items():
    X_imputed = imputer.fit_transform(X_missing)
    rmse = np.sqrt(np.mean((X_imputed[mask] - X_complete[mask])**2))
    print(f'{name:<14} {rmse:.4f}')

# ── Missing Indicator - preserve missingness information ─────────
from sklearn.impute import MissingIndicator

df_with_missing = pd.DataFrame(X_missing, columns=['feat1','feat2','feat3'])
indicator = MissingIndicator(features='missing-only')
missing_flags = indicator.fit_transform(df_with_missing)

# Add indicator columns to dataframe
flag_cols = [f'{c}_was_missing' for c in df_with_missing.columns
             if df_with_missing[c].isnull().any()]
print(f'\nAdded {missing_flags.shape[1]} missing indicator columns: {flag_cols}')